In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [25]:
import pandas as pd


file_path = '/content/drive/MyDrive/10k_processed/chunk_meta.parquet'
df = pd.read_parquet(file_path)
df.head(10)



,id,doc_id,filename,page_start,page_end,text_clean
0,0,9cf140658edcf90c20d3825c90fd9c0f,3M_10K.pdf,1,1,Table of Contents UNITED STATES SECURITIES AND...
1,1,9cf140658edcf90c20d3825c90fd9c0f,3M_10K.pdf,1,1,Yes o No x Indicate by check mark whether the ...
2,2,9cf140658edcf90c20d3825c90fd9c0f,3M_10K.pdf,1,3,o Indicate by check mark whether any of those ...
3,3,9cf140658edcf90c20d3825c90fd9c0f,3M_10K.pdf,3,3,Cybersecurity 17 Item 2. Properties 18 Item 3....
4,4,9cf140658edcf90c20d3825c90fd9c0f,3M_10K.pdf,3,3,Financial Statements and Supplementary Data 41...
5,5,9cf140658edcf90c20d3825c90fd9c0f,3M_10K.pdf,3,5,Supplemental Balance Sheet Information 61 NOTE...
6,6,9cf140658edcf90c20d3825c90fd9c0f,3M_10K.pdf,3,5,Supplier Finance Program Obligations 77 2 Tabl...
7,7,9cf140658edcf90c20d3825c90fd9c0f,3M_10K.pdf,5,5,Quarterly Data (Unaudited) 113 Item 9. Changes...
8,8,9cf140658edcf90c20d3825c90fd9c0f,3M_10K.pdf,5,6,Certain Relationships and Related Transactions...
9,9,9cf140658edcf90c20d3825c90fd9c0f,3M_10K.pdf,6,6,"In this document, for any references to Note 1..."


In [6]:

import os
import pandas as pd
from openai import OpenAI
import os
os.environ["OPENAI_API_KEY"] = "xxxxxxxxxxxxxxxxxx"
from openai import OpenAI
client = OpenAI()



In [16]:
## This block of code is reference from open AI ChatGPT
import os, time, math
import pandas as pd
from openai import OpenAI


def embed_dataframe(
    df: pd.DataFrame,
    text_col: str = "text_clean",
    model: str = "text-embedding-3-small",
    batch_size: int = 32,
    truncate_chars: int = 8000,
    out_path: str | None = None,
    save_every_batches: int = 50
) -> pd.DataFrame:
    if text_col not in df.columns:
        raise ValueError(f"Missing column: {text_col}")
    if "embedding" not in df.columns:
        df["embedding"] = None

    # skip exsiting rows
    pending = df.index[df["embedding"].isna() | df["embedding"].isnull()].tolist()
    total_batches = math.ceil(len(pending) / batch_size)
    print(f"Pending rows: {len(pending)} | batch_size={batch_size} | batches={total_batches}")

    def embed_batch(texts):
        ## rery with exp back
        delay = 1.0
        for attempt in range(6):
            try:
                resp = client.embeddings.create(model=model, input=texts)
                return [d.embedding for d in resp.data]
            except Exception as e:
                if attempt == 5:
                    raise
                time.sleep(delay)
                delay = min(delay * 2, 8.0)

    try:
        for b, start in enumerate(range(0, len(pending), batch_size), start=1):
            idxs = pending[start:start + batch_size]
            texts = [str(df.at[i, text_col])[:truncate_chars] for i in idxs]


            keep_mask = [len(t.strip()) >= 30 for t in texts]
            keep_idxs = [i for i, k in zip(idxs, keep_mask) if k]
            keep_texts = [t for t, k in zip(texts, keep_mask) if k]

            if keep_texts:
                vecs = embed_batch(keep_texts)
                for i, v in zip(keep_idxs, vecs):
                    df.at[i, "embedding"] = v

            for i, k in zip(idxs, keep_mask):
                if not k:
                    df.at[i, "embedding"] = []

            if out_path and (b % save_every_batches == 0 or b == total_batches):
                df.to_parquet(out_path, index=False)
                print(f"Checkpoint saved @ batch {b}/{total_batches}")

    except KeyboardInterrupt:
        if out_path:
            df.to_parquet(out_path, index=False)
            print("Interrupted. Progress saved.")
        raise

    return df




In [26]:
# covesrsion process
df = embed_dataframe(df, text_col="text_clean",
                      model="text-embedding-3-small",
                      batch_size=32,
                      out_path="chunks_with_embeddings.parquet")
df.to_parquet("final_embedding.parquet", index=False)

Pending rows: 8817 | batch_size=32 | batches=276
Checkpoint saved @ batch 50/276
Checkpoint saved @ batch 100/276
Checkpoint saved @ batch 150/276
Checkpoint saved @ batch 200/276
Checkpoint saved @ batch 250/276
Checkpoint saved @ batch 276/276


In [ ]:
#Check dimension
dims = df["embedding"].dropna().map(lambda v: len(v) if isinstance(v, list) else None)
print(dims.value_counts().head())  


embedding
1536    8817
Name: count, dtype: int64


In [28]:
import numpy as np

# drop rows with empty or missing embeddings
df = df[df["embedding"].apply(lambda v: isinstance(v, list) and len(v) > 0)].copy()

# L2 normalize in place
def l2(v):
    a = np.asarray(v, dtype=np.float32)
    n = np.linalg.norm(a)
    return (a / n).tolist() if n else a.tolist()

df["embedding"] = df["embedding"].map(l2)


In [29]:
dims = df["embedding"].map(len).value_counts()
print(dims)


embedding
1536    8817
Name: count, dtype: int64


In [40]:

import pandas as pd
import numpy as np
import faiss
import json, math

PARQUET_PATH = "chunks_with_embeddings.parquet"
OUT_INDEX     = "kb_hnsw_ip.faiss"
ID_COL        = None

df = pd.read_parquet(PARQUET_PATH)
print("rows:", len(df), "| has embedding:", "embedding" in df.columns)

def coerce_emb(v):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return None
    if isinstance(v, list):
        return v
    if isinstance(v, np.ndarray):
        return v.astype(np.float32).tolist()
    if isinstance(v, (bytes, bytearray)):
        try: return json.loads(v.decode("utf-8"))
        except: return None
    if isinstance(v, str):
        s = v.strip()
        if s.startswith("[") and s.endswith("]"):
            try: return json.loads(s)
            except: return None
    return None

df["embedding"] = df["embedding"].apply(coerce_emb)

mask = df["embedding"].apply(lambda v: isinstance(v, list) and len(v) > 0)
df = df[mask].copy()
if len(df) == 0:
    raise ValueError("No valid embeddings found. Check your parquet.")
print("valid vectors:", len(df))

xb = np.vstack(df["embedding"].values).astype("float32")
norms = np.linalg.norm(xb, axis=1, keepdims=True)
norms[norms == 0] = 1.0
xb = xb / norms
d = xb.shape[1]


if ID_COL and ID_COL in df.columns:
    ids = df[ID_COL].to_numpy(dtype=np.int64, copy=True)
else:
    ids = df.index.to_numpy(dtype=np.int64, copy=True)

# hnsw graph indexing for easy indexing and quering
M = 32
ef_construction = 200
ef_search = 64

base = faiss.IndexHNSWFlat(d, M, faiss.METRIC_INNER_PRODUCT)
base.hnsw.efConstruction = ef_construction
base.hnsw.efSearch = ef_search

index = faiss.IndexIDMap2(base)
index.add_with_ids(xb, ids)


faiss.write_index(index, OUT_INDEX)
print(f"Saved {OUT_INDEX} | ntotal={index.ntotal} | dim={d} | M={M} | efC={ef_construction} | efS={ef_search}")


rows: 8817 | has embedding: True
valid vectors: 8817
Saved kb_hnsw_ip.faiss | ntotal=8817 | dim=1536 | M=32 | efC=200 | efS=64


In [46]:

from sentence_transformers import CrossEncoder


reranker = CrossEncoder("BAAI/bge-reranker-base")

EMBED_MODEL  = "text-embedding-3-small"
K            = 10


df = pd.read_parquet('/content/chunks_with_embeddings.parquet')
index = faiss.read_index('/content/kb_hnsw_ip.faiss')


client = OpenAI()

def embed_query(text: str, model: str = EMBED_MODEL) -> np.ndarray:
    v = client.embeddings.create(model=model, input=[text]).data[0].embedding
    v = np.asarray(v, dtype=np.float32)

    n = np.linalg.norm(v)
    if n > 0: v = v / n
    return v.reshape(1, -1)
def search_and_show_reranked(query: str, k: int = 30, m: int = 6):
    # FAISS recall
    q = embed_query(query)
    sims, ids = index.search(q, k)
    sims, ids = sims[0], ids[0]
    candidates = []
    for _id, sim in zip(ids, sims):
        if _id == -1:
            continue
        row = df.loc[_id] if _id in df.index else df.iloc[int(_id)]
        text = str(row.get("text", row.get("text_clean", "")))
        candidates.append((_id, sim, text))

    if not candidates:
        print("No candidates found.")
        return

    # re ranking with encoder
    pairs = [(query, c[2]) for c in candidates]
    rr_scores = reranker.predict(pairs)

    # it will sort the answer by its re ranker score
    ranked = sorted(zip(candidates, rr_scores), key=lambda x: x[1], reverse=True)[:m]

    # Displayng the top 6 result from the re ranked search
    print(f"\nQuery: {query}\n")
    for rank, ((rid, faiss_sim, text), rr) in enumerate(ranked, start=1):
        row = df.loc[rid] if rid in df.index else df.iloc[int(rid)]
        title   = row.get("doc_title", row.get("filename", ""))
        section = row.get("section", "")
        page    = row.get("page_start", "")
        snippet = (text.strip().replace("\n"," ")[:300] + "…") if len(text) > 300 else text.strip()
        print(f"[{rank}] id={rid}  rerank={rr:.4f}  faiss_sim={faiss_sim:.4f}")
        print(f"    {title} · {section} · p.{page}")
        print(f"    {snippet}\n")

# top re ranked search from the document
search_and_show_reranked("Since when has Alan R. Mulally served on the company’s Board of Directors?")



Query: Since when has Alan R. Mulally served on the company’s Board of Directors?

[1] id=349  rerank=1.0000  faiss_sim=0.8400
    AIG_10K.pdf ·  · p.6
    Alan R. Mulally has served as a member of our Board of Directors since July 2014. Alan served as President and Chief Executive Officer of Ford Motor Company, a global automotive company, from September 2006 through June 2014. Alan was previously a member of the board of directors of Ford and served …

[2] id=348  rerank=0.9981  faiss_sim=0.5822
    AIG_10K.pdf ·  · p.6
    John has announced that he plans to resign from his position as the President of Stanford University in August 2016. Ann Mather has served as a member of our Board of Directors since November 2005. Ann has also been a member of the board of directors of: Arista Networks, Inc., a computer networking …

[3] id=411  rerank=0.9695  faiss_sim=0.4419
    AIG_10K.pdf ·  · p.31
    John is a trustee of the Hennessy 1993 Revocable Trust and has voting and investment autho

In [47]:
search_and_show_reranked("What major risks does Alphabet cite from operating internationally and from financial exposures?")


Query: What major risks does Alphabet cite from operating internationally and from financial exposures?

[1] id=979  rerank=0.9684  faiss_sim=0.6942
    Alphabet_10K.pdf ·  · p.19
    In addition, our products and services are highly technical and complex and have contained in the past, and may contain in the future, errors or vulnerabilities, which could result in interruptions in or failure of our services or systems. Any of these incidents could impede or prevent us from effec…

[2] id=975  rerank=0.6258  faiss_sim=0.5895
    Alphabet_10K.pdf ·  · p.18
    For example, if we fail to respond appropriately to the sharing of misinformation or objectionable content on our services and/or products or objectionable practices by advertisers, or otherwise to adequately address user concerns, our users may lose confidence in our brands. Furthermore, failure to…

[3] id=3974  rerank=0.5723  faiss_sim=0.5754
    Costco_10K.pdf ·  · p.17
    Legal and Regulatory Risks We are subject to risks a

In [48]:
search_and_show_reranked("Briefly explain how Microsoft measures fair value for Level 2 and Level 3 items, how it values equity investments without readily determinable fair values, and its policy for property and equipment.")


Query: Briefly explain how Microsoft measures fair value for Level 2 and Level 3 items, how it values equity investments without readily determinable fair values, and its policy for property and equipment.

[1] id=6522  rerank=0.9868  faiss_sim=0.6324
    Microsoft_10K.pdf ·  · p.61
    agency securities, foreign government bonds, mortgage- and asset-backed securities, corporate notes and bonds, and municipal securities. Our Level 2 derivative assets and liabilities include certain cleared swap contracts and over-the-counter forward, option, and swap contracts. • Level 3 – inputs a…

[2] id=4999  rerank=0.9785  faiss_sim=0.6567
    General_Electric_10K.pdf ·  · p.69
    Investments that are measured at fair value using the NAV practical expedient are not required to be classified in the fair value hierarchy. Investments classified within Level 3 primarily relate to real estate and private equities which are valued using unobservable inputs, primarily by discounting…

[3] id=5000  rera